In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [2]:
q1 = "Can I still join the course after the start date?"
v1 = model.encode(q1)

In [3]:
d  = "You don't need to register. You're accepted. You can also just start learning and submitting homework without registering."
dv = model.encode(d)

In [4]:
v1.dot(dv)

np.float32(0.3233239)

In [5]:
q2 = "How to install Docker on Windows?"
v2 = model.encode(q2)

In [6]:
v2.dot(dv)

np.float32(0.019730594)

In [8]:
from ingest import load_faq_data

documents = load_faq_data()

In [9]:
texts = []

for doc in documents:
    text = doc["question"] + " " + doc["answer"]
    texts.append(text)

In [11]:
texts[:5]

["How do I submit homework? - Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)",
 'What’s new in the 2025 edition? - Deployment module updated to **FastAPI** (replacing Flask) and new tools.\n- Neural networks taught with **PyTorch** (theory videos in Keras are kept; an additional PyTorch implementation video is provided).\n- Deep learning deployment uses **ONNX Runtime** on AWS Lambda (replacing TensorFlow Lite).',
 'Are Jupyter Notebooks used? Yes. You’ll work extensively with notebooks alongside standard Python files and CLI tools.',
 'Do I need prior machine l

In [12]:
from tqdm.auto import tqdm

In [13]:
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

len(vectors)

  0%|          | 0/25 [00:00<?, ?it/s]

1208

In [14]:
import numpy as np
X = np.array(vectors)

In [15]:
X

array([[-1.03578962e-01, -6.19631484e-02, -1.13746412e-02, ...,
         5.81632629e-02, -2.28137262e-02, -1.11605795e-02],
       [-1.16193160e-01, -1.01877414e-01,  5.09269461e-02, ...,
        -3.24032642e-02,  2.07023676e-02,  5.45928702e-02],
       [-7.64710009e-02,  6.32649608e-05,  3.17689851e-02, ...,
         1.90290548e-02,  8.44088122e-02, -3.48371491e-02],
       ...,
       [ 8.78644176e-03, -7.50777721e-02,  2.73207985e-02, ...,
        -5.20802010e-03,  1.72090307e-02,  3.52643169e-02],
       [-1.12957899e-02,  4.22346257e-02, -3.60510200e-02, ...,
        -3.29751186e-02, -7.11074937e-03, -1.41081074e-02],
       [-1.85938478e-02, -9.51881148e-03, -5.15211560e-02, ...,
         4.78163473e-02,  2.65053660e-02,  7.47033656e-02]],
      shape=(1208, 384), dtype=float32)

In [ ]:
# Nico: 1) is it deterministic? and 2) do we get the same vectors if we encode everything at once?

# 1) Answer is yes
batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

X = np.array(vectors)
print(X)


  0%|          | 0/25 [00:00<?, ?it/s]

[[-1.03578962e-01 -6.19631484e-02 -1.13746412e-02 ...  5.81632629e-02
  -2.28137262e-02 -1.11605795e-02]
 [-1.16193160e-01 -1.01877414e-01  5.09269461e-02 ... -3.24032642e-02
   2.07023676e-02  5.45928702e-02]
 [-7.64710009e-02  6.32649608e-05  3.17689851e-02 ...  1.90290548e-02
   8.44088122e-02 -3.48371491e-02]
 ...
 [ 8.78644176e-03 -7.50777721e-02  2.73207985e-02 ... -5.20802010e-03
   1.72090307e-02  3.52643169e-02]
 [-1.12957899e-02  4.22346257e-02 -3.60510200e-02 ... -3.29751186e-02
  -7.11074937e-03 -1.41081074e-02]
 [-1.85938478e-02 -9.51881148e-03 -5.15211560e-02 ...  4.78163473e-02
   2.65053660e-02  7.47033656e-02]]


In [17]:
X.shape

(1208, 384)

In [ ]:
# 2) answer is Yes too. And it does not take significantly much more time to encode everything at once (vs. what Alexey said)
vectors = model.encode(texts)
X = np.array(vectors)
print(X)

[[-1.0357893e-01 -6.1963193e-02 -1.1374609e-02 ...  5.8163267e-02
  -2.2813709e-02 -1.1160594e-02]
 [-1.1619316e-01 -1.0187741e-01  5.0926946e-02 ... -3.2403264e-02
   2.0702368e-02  5.4592870e-02]
 [-7.6471001e-02  6.3264961e-05  3.1768985e-02 ...  1.9029055e-02
   8.4408812e-02 -3.4837149e-02]
 ...
 [ 8.7864418e-03 -7.5077772e-02  2.7320798e-02 ... -5.2080201e-03
   1.7209031e-02  3.5264317e-02]
 [-1.1295790e-02  4.2234626e-02 -3.6051020e-02 ... -3.2975119e-02
  -7.1107494e-03 -1.4108107e-02]
 [-1.8593848e-02 -9.5188115e-03 -5.1521156e-02 ...  4.7816347e-02
   2.6505366e-02  7.4703366e-02]]


In [19]:
query = "Can I still join the course after the start date?"
v_query = model.encode(query)

In [20]:
scores = X.dot(v_query)

In [21]:
idx = np.argmax(scores)
idx, scores[idx]

(np.int64(553), np.float32(0.762941))

In [22]:
documents[idx]

{'id': '3f1424af17',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: Can I still join the course after the start date?',
 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

In [23]:
top5 = np.argsort(-scores)[:5]
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.762941
{'id': '3f1424af17', 'course': 'data-engineering-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course: Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homework.\n\nBe aware, however, that there will be deadlines for turning in homeworks and the final projects. So don't leave everything for the last minute."}

0.7579369
{'id': '2d8b16c2a0', 'course': 'mlops-zoomcamp', 'section': 'General Course-Related Questions', 'question': 'Course - Can I still join the course after the start date?', 'answer': "Yes, even if you don't register, you're still eligible to submit the homeworks as long as the form is still open and accepting submissions.\n\nBe aware, however, that there will be deadlines for turning in the final projects. So don't leave everything to the last minute."}

0.7192132
{'id': '41aabbd7c5', 'course': 'machine-learning-zoomcamp', 'section': 'General Course-Related 

In [24]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=["course"])
vindex.fit(X, documents)

In [26]:
vindex.vectors

array([[-1.0357893e-01, -6.1963193e-02, -1.1374609e-02, ...,
         5.8163267e-02, -2.2813709e-02, -1.1160594e-02],
       [-1.1619316e-01, -1.0187741e-01,  5.0926946e-02, ...,
        -3.2403264e-02,  2.0702368e-02,  5.4592870e-02],
       [-7.6471001e-02,  6.3264961e-05,  3.1768985e-02, ...,
         1.9029055e-02,  8.4408812e-02, -3.4837149e-02],
       ...,
       [ 8.7864418e-03, -7.5077772e-02,  2.7320798e-02, ...,
        -5.2080201e-03,  1.7209031e-02,  3.5264317e-02],
       [-1.1295790e-02,  4.2234626e-02, -3.6051020e-02, ...,
        -3.2975119e-02, -7.1107494e-03, -1.4108107e-02],
       [-1.8593848e-02, -9.5188115e-03, -5.1521156e-02, ...,
         4.7816347e-02,  2.6505366e-02,  7.4703366e-02]],
      shape=(1208, 384), dtype=float32)

In [27]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vindex.search(query_vector, num_results=5)

In [28]:
results[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [29]:
results = vindex.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)

In [30]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [31]:
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)

In [43]:
from rag_helper import RAGBase, RAGVector

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    model='openai/gpt-oss-20b'
)

ImportError: cannot import name 'RAGVector' from 'rag_helper' (/Users/nhubert/Documents/llm-zoomcamp-code/02-vector-search/rag_helper.py)

In [41]:
query = "I just found out about the program, can I still sign up?"
assistant.rag(query)

'Yes! You can still sign up. If you’d like to earn a certificate, be sure to submit your project while we’re still accepting submissions.'

In [44]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self, query, num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {"course": self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [47]:
vector_assistant = RAGVector(
    embedder=model,
    index=vindex,
    llm_client=openai_client,
    model='openai/gpt-oss-20b'
)

In [48]:
vector_assistant.rag("the program has already begun, can I still sign up?")

'Yes – you can still join the program. Just be sure to submit your capstone project while the submission window is still open if you want to receive a certificate.'

In [49]:
from sqlitesearch import VectorSearchIndex

vs_index = VectorSearchIndex(
    keyword_fields=["course"],
    mode="ivf",
    db_path="faq_vectors2.db"
)

In [50]:
vs_index.fit(vectors, documents)

In [51]:
query = "I just discovered the course. Can I still join it?"
query_vector = model.encode(query)

results = vs_index.search(query_vector, num_results=5)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '41aabbd7c5',
  'course': 'machine-learning-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The course has already started. Can I still join it?',
  'answer': 'Yes, you can. Even though you missed the start date, you can register for the course. You won’t be able to submit some of the homeworks, but you can still take part in the course.\n\nIn order to get a certificate, you need to submit 2 out of 3 course projects and review 3 peers by the deadline. It means that if you join the course at the end of November and manage to work on two projects, you will still be eligible for a certificate.'},
 {'id': '2d8b16c2a0',
  'course': 'mlops-zoomcamp',
  'section':

In [52]:
results = vs_index.search(
    query_vector,
    filter_dict={"course": "llm-zoomcamp"},
    num_results=5
)
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time the course is running; after the form is closed and the peer-review list is compiled.'},
 {'id': 'bd31146b0e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'When will the co

In [53]:
vs_index.close()